# RAG Sufficient Context — Single Pipeline

Two execution modes:
- **`portable_verified`** — synthetic dataset + deterministic mock generation. Runs without GPU or model downloads.
- **`full_hf_auto`** — HotPotQA + real Hugging Face model. Falls back to `portable_verified` on any error.

All heavy logic (retrieval, evaluation, gate, calibration, plotting) comes from `src`.
Only the dataset fixture and mock generation functions live in this notebook.

In [6]:
CONFIG = {
    "execution": {
        "profile": "full_hf_auto",  # portable_verified | full_hf_auto
        "output_dir": "../results/single_notebook_run",
        "seed": 42,
    },
    "dataset": {
        "num_examples": 200,
        "top_k": 2,
        "max_context_tokens": 160,
    },
    "generation": {
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "max_new_tokens": 96,
        "temperature": 0.0,
    },
    "autorater": {
        "chunk_size_tokens": 120,
        "aggregation": "support_all_required",
        "max_new_tokens": 64,
    },
    "confidence": {
        "method": "inline",  # inline | token_entropy | self_report
    },
    "gate": {
        "threshold_steps": 50,
    },
    "evaluation": {
        "f1_threshold": 0.5,
    },
}

print(CONFIG)

{'execution': {'profile': 'full_hf_auto', 'output_dir': 'results/single_notebook_run', 'seed': 42}, 'dataset': {'num_examples': 200, 'top_k': 2, 'max_context_tokens': 160}, 'generation': {'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'max_new_tokens': 96, 'temperature': 0.0}, 'autorater': {'chunk_size_tokens': 120, 'aggregation': 'support_all_required', 'max_new_tokens': 64}, 'confidence': {'method': 'inline'}, 'gate': {'threshold_steps': 50}, 'evaluation': {'f1_threshold': 0.5}}


In [7]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np

# --- src imports ---
sys.path.insert(0, str(Path.cwd().parent))  # make src importable when running from notebooks/

from src.utils import set_global_seed, save_run_metadata
from src.retrieval import build_retrieval_pipeline, load_hotpotqa, summarize_retrieval_metrics
from src.evaluation import evaluate_all
from src.gate import (
    compute_selective_curves,
    plot_accuracy_coverage,
    plot_sufficiency_breakdown,
    plot_calibration_curve,
    plot_score_distributions,
    plot_support_recall_vs_f1,
)

print("src imports OK")

src imports OK


In [8]:
# ---------------------------------------------------------------------------
# Mock helpers — test fixtures for portable_verified mode.
# These functions live in the notebook because they are test scaffolding,
# not production logic. All analysis code uses src.
# ---------------------------------------------------------------------------

def _stable_hash(text: str) -> int:
    value = 0
    for ch in text:
        value = (value * 131 + ord(ch)) % 10_000_019
    return value


def synthetic_examples():
    """18-example synthetic dataset that covers bridge/comparison and easy/medium/hard."""
    return [
        {"id": "ex01", "question": "What is the capital of the country where the Danube begins?",
         "answer": "Berlin", "type": "bridge", "level": "hard",
         "supporting_fact_titles": ["Danube", "Germany"],
         "passages": [
             {"title": "Danube", "text": "Danube: The Danube river begins in Germany, in the Black Forest region."},
             {"title": "Germany", "text": "Germany: Germany is a country in Europe. Its capital city is Berlin."},
             {"title": "Austria", "text": "Austria: The Danube flows through Austria. Vienna is the capital of Austria."},
             {"title": "Black Forest", "text": "Black Forest: The Black Forest is a forested mountain range in southwest Germany."},
         ]},
        {"id": "ex02", "question": "Which author was born earlier, Jane Austen or George Orwell?",
         "answer": "Jane Austen", "type": "comparison", "level": "easy",
         "supporting_fact_titles": ["Jane Austen", "George Orwell"],
         "passages": [
             {"title": "Jane Austen", "text": "Jane Austen: Jane Austen was born in 1775 in Hampshire, England."},
             {"title": "George Orwell", "text": "George Orwell: George Orwell was born in 1903 in Motihari, British India."},
             {"title": "Virginia Woolf", "text": "Virginia Woolf: Virginia Woolf was born in 1882."},
             {"title": "Charles Dickens", "text": "Charles Dickens: Charles Dickens was born in 1812."},
         ]},
        {"id": "ex03", "question": "What city is the capital of the country that uses the yen?",
         "answer": "Tokyo", "type": "bridge", "level": "easy",
         "supporting_fact_titles": ["Yen", "Japan"],
         "passages": [
             {"title": "Yen", "text": "Yen: The yen is the official currency of Japan."},
             {"title": "Japan", "text": "Japan: Japan is an island country in East Asia. Its capital is Tokyo."},
             {"title": "China", "text": "China: The renminbi is the official currency of China. Beijing is its capital."},
             {"title": "Tokyo", "text": "Tokyo: Tokyo is the capital city of Japan."},
         ]},
        {"id": "ex04", "question": "Which mountain is higher, Mount Kilimanjaro or Mount Elbrus?",
         "answer": "Mount Kilimanjaro", "type": "comparison", "level": "medium",
         "supporting_fact_titles": ["Mount Kilimanjaro", "Mount Elbrus"],
         "passages": [
             {"title": "Mount Kilimanjaro", "text": "Mount Kilimanjaro: Mount Kilimanjaro has an elevation of 5,895 metres."},
             {"title": "Mount Elbrus", "text": "Mount Elbrus: Mount Elbrus has an elevation of 5,642 metres."},
             {"title": "Mount Kenya", "text": "Mount Kenya: Mount Kenya is 5,199 metres high."},
             {"title": "Africa", "text": "Africa: Mount Kilimanjaro is the highest mountain in Africa."},
         ]},
        {"id": "ex05", "question": "What language is primarily spoken in the country whose capital is Brasilia?",
         "answer": "Portuguese", "type": "bridge", "level": "medium",
         "supporting_fact_titles": ["Brasilia", "Brazil"],
         "passages": [
             {"title": "Brasilia", "text": "Brasilia: Brasilia is the capital of Brazil."},
             {"title": "Brazil", "text": "Brazil: Portuguese is the primary language spoken in Brazil."},
             {"title": "Argentina", "text": "Argentina: Buenos Aires is the capital of Argentina and Spanish is spoken there."},
             {"title": "Portuguese language", "text": "Portuguese language: Portuguese is a Romance language spoken in Portugal and Brazil."},
         ]},
        {"id": "ex06", "question": "Which scientist died later, Isaac Newton or Albert Einstein?",
         "answer": "Albert Einstein", "type": "comparison", "level": "easy",
         "supporting_fact_titles": ["Isaac Newton", "Albert Einstein"],
         "passages": [
             {"title": "Isaac Newton", "text": "Isaac Newton: Isaac Newton died in 1727."},
             {"title": "Albert Einstein", "text": "Albert Einstein: Albert Einstein died in 1955."},
             {"title": "Galileo Galilei", "text": "Galileo Galilei: Galileo died in 1642."},
             {"title": "Physics", "text": "Physics: Newton and Einstein both contributed to physics."},
         ]},
        {"id": "ex07", "question": "What is the capital of the country where the Amazon River flows?",
         "answer": "Brasilia", "type": "bridge", "level": "hard",
         "supporting_fact_titles": ["Amazon River", "Brazil"],
         "passages": [
             {"title": "Amazon River", "text": "Amazon River: The Amazon River flows through Brazil, Peru, and Colombia."},
             {"title": "Brazil", "text": "Brazil: Brazil is a country in South America whose capital is Brasilia."},
             {"title": "Peru", "text": "Peru: Peru is a South American country. Its capital is Lima."},
             {"title": "Colombia", "text": "Colombia: Colombia is a South American country. Its capital is Bogota."},
         ]},
        {"id": "ex08", "question": "Which city is farther north, Rome or Madrid?",
         "answer": "Rome", "type": "comparison", "level": "medium",
         "supporting_fact_titles": ["Rome", "Madrid"],
         "passages": [
             {"title": "Rome", "text": "Rome: Rome lies at about 41.9 degrees north latitude."},
             {"title": "Madrid", "text": "Madrid: Madrid lies at about 40.4 degrees north latitude."},
             {"title": "Italy", "text": "Italy: Rome is the capital of Italy."},
             {"title": "Spain", "text": "Spain: Madrid is the capital of Spain."},
         ]},
        {"id": "ex09", "question": "What animal is the symbol of the country whose capital is Canberra?",
         "answer": "Kangaroo", "type": "bridge", "level": "hard",
         "supporting_fact_titles": ["Canberra", "Australia"],
         "passages": [
             {"title": "Canberra", "text": "Canberra: Canberra is the capital city of Australia."},
             {"title": "Australia", "text": "Australia: The kangaroo is a widely recognized symbol of Australia."},
             {"title": "Emu", "text": "Emu: The emu also appears on the coat of arms of Australia."},
             {"title": "Sydney", "text": "Sydney: Sydney is a major city in Australia but not the capital."},
         ]},
        {"id": "ex10", "question": "Which country has the larger population, Canada or Australia?",
         "answer": "Canada", "type": "comparison", "level": "easy",
         "supporting_fact_titles": ["Canada", "Australia"],
         "passages": [
             {"title": "Canada", "text": "Canada: Canada has a population of about 40 million people."},
             {"title": "Australia", "text": "Australia: Australia has a population of about 27 million people."},
             {"title": "Ottawa", "text": "Ottawa: Ottawa is the capital of Canada."},
             {"title": "Canberra", "text": "Canberra: Canberra is the capital of Australia."},
         ]},
        {"id": "ex11", "question": "Which instrument is associated with the composer who wrote The Four Seasons?",
         "answer": "Violin", "type": "bridge", "level": "medium",
         "supporting_fact_titles": ["The Four Seasons", "Antonio Vivaldi"],
         "passages": [
             {"title": "The Four Seasons", "text": "The Four Seasons: The Four Seasons is a group of violin concertos by Antonio Vivaldi."},
             {"title": "Antonio Vivaldi", "text": "Antonio Vivaldi: Vivaldi was an Italian composer and violinist."},
             {"title": "Piano", "text": "Piano: The piano is a keyboard instrument."},
             {"title": "Baroque", "text": "Baroque: Vivaldi composed during the Baroque era."},
         ]},
        {"id": "ex12", "question": "Which city is the capital of the country that borders Portugal to the east?",
         "answer": "Madrid", "type": "bridge", "level": "medium",
         "supporting_fact_titles": ["Portugal", "Spain"],
         "passages": [
             {"title": "Portugal", "text": "Portugal: Portugal is bordered by Spain to the east and north."},
             {"title": "Spain", "text": "Spain: Spain is a country in Europe whose capital city is Madrid."},
             {"title": "Lisbon", "text": "Lisbon: Lisbon is the capital city of Portugal."},
             {"title": "Iberian Peninsula", "text": "Iberian Peninsula: Spain and Portugal are located on the Iberian Peninsula."},
         ]},
        {"id": "ex13", "question": "Which planet is larger, Earth or Mars?",
         "answer": "Earth", "type": "comparison", "level": "easy",
         "supporting_fact_titles": ["Earth", "Mars"],
         "passages": [
             {"title": "Earth", "text": "Earth: Earth has a diameter of about 12,742 kilometres."},
             {"title": "Mars", "text": "Mars: Mars has a diameter of about 6,779 kilometres."},
             {"title": "Solar System", "text": "Solar System: Earth and Mars are planets in the Solar System."},
             {"title": "Jupiter", "text": "Jupiter: Jupiter is the largest planet in the Solar System."},
         ]},
        {"id": "ex14", "question": "What drink is traditionally associated with the country whose capital is Dublin?",
         "answer": "Tea", "type": "bridge", "level": "hard",
         "supporting_fact_titles": ["Dublin", "Ireland"],
         "passages": [
             {"title": "Dublin", "text": "Dublin: Dublin is the capital city of Ireland."},
             {"title": "Ireland", "text": "Ireland: Tea is a very common and traditional drink in Ireland."},
             {"title": "Guinness", "text": "Guinness: Guinness is a famous Irish stout."},
             {"title": "Belfast", "text": "Belfast: Belfast is a city on the island of Ireland."},
         ]},
        {"id": "ex15", "question": "Which country is larger by area, India or Pakistan?",
         "answer": "India", "type": "comparison", "level": "medium",
         "supporting_fact_titles": ["India", "Pakistan"],
         "passages": [
             {"title": "India", "text": "India: India has an area of about 3.287 million square kilometres."},
             {"title": "Pakistan", "text": "Pakistan: Pakistan has an area of about 881,913 square kilometres."},
             {"title": "South Asia", "text": "South Asia: India and Pakistan are countries in South Asia."},
             {"title": "Delhi", "text": "Delhi: New Delhi is the capital of India."},
         ]},
        {"id": "ex16", "question": "What is the capital of the country where Mount Fuji is located?",
         "answer": "Tokyo", "type": "bridge", "level": "easy",
         "supporting_fact_titles": ["Mount Fuji", "Japan"],
         "passages": [
             {"title": "Mount Fuji", "text": "Mount Fuji: Mount Fuji is the highest mountain in Japan."},
             {"title": "Japan", "text": "Japan: The capital city of Japan is Tokyo."},
             {"title": "Kyoto", "text": "Kyoto: Kyoto was once the imperial capital of Japan."},
             {"title": "Asia", "text": "Asia: Japan is located in East Asia."},
         ]},
        {"id": "ex17", "question": "Which writer lived longer, Leo Tolstoy or Fyodor Dostoevsky?",
         "answer": "Leo Tolstoy", "type": "comparison", "level": "hard",
         "supporting_fact_titles": ["Leo Tolstoy", "Fyodor Dostoevsky"],
         "passages": [
             {"title": "Leo Tolstoy", "text": "Leo Tolstoy: Leo Tolstoy lived from 1828 to 1910."},
             {"title": "Fyodor Dostoevsky", "text": "Fyodor Dostoevsky: Fyodor Dostoevsky lived from 1821 to 1881."},
             {"title": "Russia", "text": "Russia: Tolstoy and Dostoevsky were Russian writers."},
             {"title": "War and Peace", "text": "War and Peace: War and Peace is a novel by Leo Tolstoy."},
         ]},
        {"id": "ex18", "question": "What metal is used in the structure located in the country whose capital is Paris?",
         "answer": "Iron", "type": "bridge", "level": "medium",
         "supporting_fact_titles": ["Paris", "Eiffel Tower"],
         "passages": [
             {"title": "Paris", "text": "Paris: Paris is the capital city of France."},
             {"title": "Eiffel Tower", "text": "Eiffel Tower: The Eiffel Tower in Paris is made primarily of wrought iron."},
             {"title": "France", "text": "France: France is a country in Europe."},
             {"title": "Steel", "text": "Steel: Many modern towers are made of steel."},
         ]},
    ]


def _mock_wrong_answer(example):
    """Return a plausible-but-wrong answer for hallucination simulation."""
    context = example.get("context", "")
    question = example["question"].lower()
    if "capital" in question:
        for city in ["Vienna", "Lima", "Bogota", "Lisbon", "Kyoto", "Sydney", "Beijing"]:
            if city.lower() in context.lower() and city.lower() not in example["answer"].lower():
                return city
        return "Paris"
    distractors = [
        t for t in example.get("retrieved_titles", [])
        if t not in example.get("supporting_fact_titles", [])
    ]
    return distractors[0] if distractors else "Unknown"


def mock_generate_answers_batch(examples):
    """Deterministic generation mock. Simulates a realistic error pattern:
    - full-support examples mostly correct, some hallucinate
    - partial-support examples mix of correct/hallucinate/abstain
    - no-support examples mostly abstain or hallucinate
    """
    out = []
    for ex in examples:
        full = bool(ex.get("full_support_coverage_post_truncation", False))
        recall = float(ex.get("support_title_recall_post_truncation", 0.0))
        key = _stable_hash(ex["id"]) % 10
        level = ex.get("level", "medium")

        if full:
            if level == "hard" and key == 1:
                answer, confidence = "I don't know", 0.18
            elif key == 7:
                answer, confidence = _mock_wrong_answer(ex), 0.74
            else:
                conf = 0.92 if level == "easy" else 0.84 if level == "medium" else 0.78
                answer, confidence = ex["answer"], conf
        elif recall >= 0.5:
            if ex.get("type") == "bridge":
                if key % 2 == 0:
                    answer, confidence = _mock_wrong_answer(ex), 0.76
                else:
                    answer, confidence = "I don't know", 0.14
            else:
                if key in {2, 3}:
                    answer, confidence = ex["answer"], 0.61
                else:
                    answer, confidence = "I don't know", 0.17
        else:
            if key in {0, 1}:
                answer, confidence = _mock_wrong_answer(ex), 0.68
            else:
                answer, confidence = "I don't know", 0.06

        out.append({
            **ex,
            "prediction": answer,
            "confidence": confidence,
            "token_entropy_proxy": None,
            "mean_token_probability": None,
        })
    return out


def mock_rate_sufficiency_batch(examples, chunk_size_tokens=120, aggregation="support_all_required"):
    """Title-overlap mock autorater. Checks whether retrieved passage groups
    cover supporting fact titles — a valid proxy when no LLM is available.
    """
    from src.utils import count_tokens
    out = []
    for ex in examples:
        passages = ex.get("retrieved_passages", [])
        support_titles = set(ex.get("supporting_fact_titles", []))

        # group passages into token-bounded chunks
        groups, current, current_tokens = [], [], 0
        for p in passages:
            tok = count_tokens(p["text"], tokenizer=None)
            if current and current_tokens + tok > chunk_size_tokens:
                groups.append(current)
                current, current_tokens = [], 0
            current.append(p)
            current_tokens += tok
        if current:
            groups.append(current)

        records = []
        for idx, group in enumerate(groups):
            titles = [p["title"] for p in group]
            sufficient_group = bool(set(titles) & support_titles)
            records.append({
                "segment_id": idx,
                "segment_type": "passage_group",
                "segment_titles": titles,
                "sufficient": sufficient_group,
                "reason": "supporting title present" if sufficient_group else "no supporting title",
                "parsed": True,
            })

        positive_titles = sorted({
            t for r in records if r["sufficient"] for t in r["segment_titles"]
        })
        if aggregation == "support_all_required":
            sufficient = support_titles.issubset(set(positive_titles)) if support_titles else False
        else:
            sufficient = any(r["sufficient"] for r in records)

        n = len(records)
        out.append({
            **ex,
            "sufficient": sufficient,
            "passage_ratings": records,
            "num_chunks": n,
            "positive_chunk_ratio": sum(r["sufficient"] for r in records) / n if n else 0.0,
            "positive_passage_titles": positive_titles,
        })
    return out


print("Mock helpers defined")

Mock helpers defined


In [9]:
set_global_seed(CONFIG["execution"]["seed"])
OUTPUT_DIR = Path(CONFIG["execution"]["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve())

Output dir: C:\Users\karim\labsubmissioms\rag-sufficient-context\notebooks\results\single_notebook_run


In [10]:
EXECUTION_MODE = CONFIG["execution"]["profile"]
DATASET_NAME = None

if EXECUTION_MODE == "portable_verified":
    examples = synthetic_examples()[:CONFIG["dataset"]["num_examples"]]
    DATASET_NAME = "synthetic_portable"
    print("Mode: portable_verified — using synthetic dataset")
else:
    try:
        examples = load_hotpotqa(
            split="validation",
            num_examples=CONFIG["dataset"]["num_examples"],
            seed=CONFIG["execution"]["seed"],
        )
        DATASET_NAME = "hotpotqa_validation"
        print("Mode: full_hf_auto — loaded HotPotQA")
    except Exception as e:
        print(f"HotPotQA load failed ({type(e).__name__}: {str(e)[:200]}); falling back to portable_verified")
        examples = synthetic_examples()[:CONFIG["dataset"]["num_examples"]]
        DATASET_NAME = "synthetic_fallback"
        EXECUTION_MODE = "portable_verified"

print(f"Dataset: {DATASET_NAME}, examples: {len(examples)}")
print(f"Sample: {examples[0]['question']}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Mode: full_hf_auto — loaded HotPotQA
Dataset: hotpotqa_validation, examples: 200
Sample: What nationality was Oliver Reed's character in the film Royal Flash?


## 2. BM25 Retrieval

In [11]:
# src.retrieval.build_retrieval_pipeline works for both synthetic and real examples.
# tokenizer=None uses approximate word-count truncation.
retrieved_examples = build_retrieval_pipeline(
    examples,
    top_k=CONFIG["dataset"]["top_k"],
    max_context_tokens=CONFIG["dataset"]["max_context_tokens"],
    tokenizer=None,
)

retrieval_summary = summarize_retrieval_metrics(retrieved_examples)
with open(OUTPUT_DIR / "retrieval_summary.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_summary, f, indent=2, ensure_ascii=False)

print(json.dumps(retrieval_summary, indent=2, ensure_ascii=False))

Retrieving contexts: 100%|██████████| 200/200 [00:00<00:00, 666.12it/s]


{
  "num_examples": 200,
  "truncation_rate": 0.0,
  "mean_context_tokens_before_truncation": 170.61,
  "mean_context_tokens_after_truncation": 170.61,
  "mean_selected_passages": 2.0,
  "support_title_recall_at_k_pre_truncation": 0.61,
  "support_title_recall_at_k_post_truncation": 0.61,
  "full_support_coverage_at_k_pre_truncation": 0.32,
  "full_support_coverage_at_k_post_truncation": 0.32,
  "lost_support_after_truncation_rate": 0.0,
  "mean_top1_bm25_score": 9.20959201605487,
  "mean_topk_bm25_score": 7.8135832413911235
}


## 3. Load Backend (Model or Mock)

In [12]:
BACKEND = None  # will be set below

if EXECUTION_MODE == "portable_verified":
    BACKEND = {"kind": "mock", "name": "mock_portable_backend", "model": None, "tokenizer": None}
    print("Backend: mock (portable_verified mode)")
else:
    try:
        from src.generation import load_model
        model, tokenizer = load_model(model_config=CONFIG["generation"])
        BACKEND = {"kind": "hf", "name": CONFIG["generation"]["model_name"], "model": model, "tokenizer": tokenizer}
        print(f"Backend: HF model '{BACKEND['name']}' loaded")
    except Exception as e:
        print(f"Model load failed ({type(e).__name__}: {str(e)[:200]}); falling back to mock backend")
        BACKEND = {"kind": "mock", "name": "mock_fallback_backend", "model": None, "tokenizer": None}
        EXECUTION_MODE = "portable_verified"
        print("Switched to portable_verified mode")

print(f"Active execution mode: {EXECUTION_MODE}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Backend: HF model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' loaded
Active execution mode: full_hf_auto


## 4. Generation

In [13]:
if BACKEND["kind"] == "hf":
    from src.generation import generate_answers_batch
    generated_examples = generate_answers_batch(
        retrieved_examples,
        BACKEND["model"],
        BACKEND["tokenizer"],
        max_new_tokens=CONFIG["generation"]["max_new_tokens"],
        temperature=CONFIG["generation"]["temperature"],
    )
else:
    generated_examples = mock_generate_answers_batch(retrieved_examples)

ex0 = generated_examples[0]
print(f"Q:          {ex0['question']}")
print(f"Gold:       {ex0['answer']}")
print(f"Prediction: {ex0['prediction']}")
print(f"Confidence: {ex0['confidence']:.3f}")

Generating answers: 100%|██████████| 200/200 [5:34:32<00:00, 100.36s/it]

Q:          What nationality was Oliver Reed's character in the film Royal Flash?
Gold:       Prussian
Prediction: Answer: Oliver Reed's character in the film Royal Flash was played by actor Oliver Reed.
Confidence: 0.100


## 5. Sufficiency Autorater

In [14]:
if BACKEND["kind"] == "hf":
    from src.autorater import rate_all_examples
    rated_examples = rate_all_examples(
        generated_examples,
        BACKEND["model"],
        BACKEND["tokenizer"],
        chunk_size_tokens=CONFIG["autorater"]["chunk_size_tokens"],
        aggregation=CONFIG["autorater"]["aggregation"],
        max_new_tokens=CONFIG["autorater"]["max_new_tokens"],
    )
else:
    rated_examples = mock_rate_sufficiency_batch(
        generated_examples,
        chunk_size_tokens=CONFIG["autorater"]["chunk_size_tokens"],
        aggregation=CONFIG["autorater"]["aggregation"],
    )

n_suf = sum(1 for ex in rated_examples if ex.get("sufficient"))
n_tot = len(rated_examples)
print(f"Sufficient:   {n_suf}/{n_tot} ({n_suf/n_tot:.1%})")
print(f"Insufficient: {n_tot - n_suf}/{n_tot} ({(n_tot - n_suf)/n_tot:.1%})")

Rating context sufficiency:   0%|          | 0/200 [00:00<?, ?it/s]Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Rating context sufficiency:   0%|          | 1/200 [02:43<9:03:25, 163.84s/it]Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take pre

Sufficient:   61/200 (30.5%)
Insufficient: 139/200 (69.5%)


## 6. Confidence Estimation

In [15]:
method = CONFIG["confidence"]["method"]

if BACKEND["kind"] == "hf" and method not in ("inline", "token_entropy"):
    from src.confidence import estimate_confidence_batch
    confidence_examples = estimate_confidence_batch(
        rated_examples,
        BACKEND["model"],
        BACKEND["tokenizer"],
        method=method,
    )
else:
    # inline: confidence already in the output; token_entropy: derived from generation stats
    from src.confidence import estimate_confidence_batch
    confidence_examples = estimate_confidence_batch(
        rated_examples,
        model=None,
        tokenizer=None,
        use_inline=(method != "token_entropy"),
        method=method if method in ("inline", "token_entropy") else "inline",
    )

print(f"Confidence method: {method}")
print(f"Example confidence values: {[round(ex['confidence'], 3) for ex in confidence_examples[:5]]}")

Estimating confidence (inline): 100%|██████████| 200/200 [00:00<00:00, 2930.47it/s]

Confidence method: inline
Example confidence values: [0.1, 0.1, 0.1, 0.1, 0.1]


## 7. Evaluation

In [16]:
evaluation = evaluate_all(
    confidence_examples,
    f1_threshold=CONFIG["evaluation"]["f1_threshold"],
)

metrics = evaluation["metrics"]
print(f"Total:       {metrics['total']}")
print(f"Correct:     {metrics['correct']} ({metrics['correct_rate']:.1%})")
print(f"Abstain:     {metrics['abstain']} ({metrics['abstain_rate']:.1%})")
print(f"Hallucinate: {metrics['hallucinate']} ({metrics['hallucinate_rate']:.1%})")
print(f"Mean F1:     {metrics['mean_f1']:.3f}")
print(f"Mean EM:     {metrics['mean_em']:.3f}")
print(f"Answered accuracy:       {metrics['answered_accuracy']:.3f}")
print(f"Safe abstention rate:    {metrics['safe_abstention_rate']:.3f}")
print(f"Unsafe answer rate:      {metrics['unsafe_answer_rate']:.3f}")

# Merge per-example eval fields back so gate/plot functions have 'category'
row_by_id = {row["id"]: row for row in evaluation["per_example"]}
examples_for_gate = [
    {**ex, **row_by_id[ex["id"]]}
    for ex in confidence_examples
    if ex["id"] in row_by_id
]

Total:       200
Correct:     6 (3.0%)
Abstain:     8 (4.0%)
Hallucinate: 186 (93.0%)
Mean F1:     0.114
Mean EM:     0.000
Answered accuracy:       0.031
Safe abstention rate:    0.043
Unsafe answer rate:      0.928


## 8. Selective Generation Gate

In [17]:
curves = compute_selective_curves(
    examples_for_gate,
    threshold_steps=CONFIG["gate"]["threshold_steps"],
)

print("Gate meta:", json.dumps(curves["gate_meta"], indent=2))
print(f"Answered rows: {curves['num_answered_rows']}")
print(f"Class balance: {curves['class_balance_answered']}")
if curves["baseline"].get("aurc") is not None:
    print(f"Baseline AURC:   {curves['baseline']['aurc']:.4f}")
if curves["proposed"].get("aurc") is not None:
    print(f"Proposed AURC:   {curves['proposed']['aurc']:.4f}")
print(f"Calibration: {json.dumps(curves['calibration'], indent=2)}")

Gate meta: {
  "status": "cross_validated",
  "cv_folds": 5,
  "class_counts": {
    "0": 186,
    "1": 6
  },
  "fallback_used": false
}
Answered rows: 192
Class balance: {'correct': 6, 'hallucinate': 186}
Proposed AURC:   0.9394
Calibration: {
  "confidence_only": {
    "ece": 0.06874999999999999,
    "brier": 0.03500000000000001,
    "auroc": 0.5,
    "auprc": 0.03125
  },
  "gate_score": {
    "ece": 1.4178359539318564e-05,
    "brier": 0.030486498102500186,
    "auroc": 0.2674731182795699,
    "auprc": 0.022321337749947844
  }
}


## 9. Plots

In [18]:
# Plot 1: Selective Accuracy vs Coverage
# X-axis: fraction of all examples that pass the threshold (end-to-end coverage)
# Y-axis: accuracy among those kept examples
# Two curves: confidence-only baseline vs gate score (confidence + sufficiency + chunk ratio + retrieval recall)
plot_accuracy_coverage(curves, save_path=str(OUTPUT_DIR / "accuracy_coverage.png"))
print("Saved: accuracy_coverage.png")

Saved: accuracy_coverage.png


In [19]:
# Plot 2: Response category breakdown by context sufficiency
# Shows what fraction of sufficient/insufficient contexts lead to correct/abstain/hallucinate
plot_sufficiency_breakdown(examples_for_gate, save_path=str(OUTPUT_DIR / "sufficiency_breakdown.png"))
print("Saved: sufficiency_breakdown.png")

Saved: sufficiency_breakdown.png


In [20]:
# Plot 3: Reliability diagrams (calibration curves)
# Requires at least 2 non-empty bins AND both classes in labels — skipped otherwise
conf_scores = curves.get("confidence_scores", [])
gate_scores = curves.get("gate_scores", [])
labels = curves.get("labels", [])

plot_calibration_curve(
    conf_scores, labels,
    title="Confidence-only Calibration",
    save_path=str(OUTPUT_DIR / "calibration_confidence.png"),
)
plot_calibration_curve(
    gate_scores, labels,
    title="Gate Score Calibration",
    save_path=str(OUTPUT_DIR / "calibration_gate.png"),
)
print("Calibration plots done")

Skipping calibration plot 'Confidence-only Calibration': fewer than 2 non-empty bins.
Skipping calibration plot 'Gate Score Calibration': fewer than 2 non-empty bins.
Calibration plots done


In [21]:
# Plot 4: Confidence score distribution by response category
plot_score_distributions(
    examples_for_gate,
    score_key="confidence",
    save_path=str(OUTPUT_DIR / "confidence_distribution.png"),
)
print("Saved: confidence_distribution.png")

Saved: confidence_distribution.png


In [22]:
# Plot 5: Support title recall (post-truncation) vs F1
# Shows whether better retrieval (higher support recall) correlates with higher F1
plot_support_recall_vs_f1(
    evaluation["per_example"],
    save_path=str(OUTPUT_DIR / "support_recall_vs_f1.png"),
)
print("Saved: support_recall_vs_f1.png")

Saved: support_recall_vs_f1.png


## 10. Save Results

In [23]:
with open(OUTPUT_DIR / "evaluation.json", "w", encoding="utf-8") as f:
    json.dump(evaluation, f, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / "selective_curves.json", "w", encoding="utf-8") as f:
    json.dump(curves, f, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / "examples_enriched.json", "w", encoding="utf-8") as f:
    json.dump(examples_for_gate, f, indent=2, ensure_ascii=False)

# run_metadata via src.utils — consistent with main_pipeline.ipynb
save_run_metadata(
    output_dir=str(OUTPUT_DIR),
    config=CONFIG,
    model_name=BACKEND["name"],
    extra={
        "execution_mode": EXECUTION_MODE,
        "dataset_name": DATASET_NAME,
        "num_examples": metrics["total"],
        "correct": metrics["correct"],
        "abstain": metrics["abstain"],
        "hallucinate": metrics["hallucinate"],
        "gate_meta": curves["gate_meta"],
    },
)

artifacts = sorted(p.name for p in OUTPUT_DIR.iterdir())
print("Artifacts:", artifacts)

Artifacts: ['accuracy_coverage.png', 'confidence_distribution.png', 'evaluation.json', 'examples_enriched.json', 'retrieval_summary.json', 'run_metadata.json', 'selective_curves.json', 'sufficiency_breakdown.png', 'support_recall_vs_f1.png']


In [24]:
summary = {
    "execution_mode": EXECUTION_MODE,
    "dataset": DATASET_NAME,
    "backend": BACKEND["name"],
    "output_dir": str(OUTPUT_DIR),
    "metrics": {
        "total": metrics["total"],
        "correct_rate": round(metrics["correct_rate"], 4),
        "abstain_rate": round(metrics["abstain_rate"], 4),
        "hallucinate_rate": round(metrics["hallucinate_rate"], 4),
        "mean_f1": round(metrics["mean_f1"], 4),
        "answered_accuracy": round(metrics["answered_accuracy"], 4),
        "safe_abstention_rate": round(metrics["safe_abstention_rate"], 4),
        "unsafe_answer_rate": round(metrics["unsafe_answer_rate"], 4),
    },
    "gate": {
        "status": curves["gate_meta"].get("status"),
        "fallback_used": curves["gate_meta"].get("fallback_used"),
        "baseline_aurc": curves["baseline"].get("aurc"),
        "proposed_aurc": curves["proposed"].get("aurc"),
    },
    "calibration": curves["calibration"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "execution_mode": "full_hf_auto",
  "dataset": "hotpotqa_validation",
  "backend": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "output_dir": "results\\single_notebook_run",
  "metrics": {
    "total": 200,
    "correct_rate": 0.03,
    "abstain_rate": 0.04,
    "hallucinate_rate": 0.93,
    "mean_f1": 0.1143,
    "answered_accuracy": 0.0312,
    "safe_abstention_rate": 0.0432,
    "unsafe_answer_rate": 0.9281
  },
  "gate": {
    "status": "cross_validated",
    "fallback_used": false,
    "baseline_aurc": null,
    "proposed_aurc": 0.9393810691534918
  },
  "calibration": {
    "confidence_only": {
      "ece": 0.06874999999999999,
      "brier": 0.03500000000000001,
      "auroc": 0.5,
      "auprc": 0.03125
    },
    "gate_score": {
      "ece": 1.4178359539318564e-05,
      "brier": 0.030486498102500186,
      "auroc": 0.2674731182795699,
      "auprc": 0.022321337749947844
    }
  }
}
